# Rerun every result in the paper

Click **Runtime → Run all**. It takes about 1–1.5 hours. Keep this tab open; if Colab disconnects, run all again.

The last cell puts your results next to the numbers reported in the paper. Small differences in the third decimal are normal if package versions differ.

## 1. Get the code and data

In [ ]:
%cd /content
!rm -rf tax-signals-fraud-replication FraudDetection
!git clone -q https://github.com/lulumypb-eng/tax-signals-fraud-replication
!git clone -q https://github.com/JarFraud/FraudDetection.git
!pip install -q imbalanced-learn xgboost shap statsmodels
%cd /content/tax-signals-fraud-replication
!ls

## 2. Main sample, 1991–2014 (Tables 1–5 and 8, Figures 1, 3, 4)

In [ ]:
!mkdir -p out
!python analysis.py /content/FraudDetection/data_FraudDetection_JAR2020.csv out

In [ ]:
!python regress.py out

In [ ]:
!python explain.py out

In [ ]:
!python figs.py out

## 3. SEC sample, 2009–2022 (Tables 6–7, Figure 2)

In [ ]:
%cd /content/tax-signals-fraud-replication/extension_2009_2022
!mkdir -p out
!python analysis_post2017.py data/sec_panel.csv.gz data/labels.csv out
!python fig_ext.py out
%cd /content/tax-signals-fraud-replication

## 4. Compare with the paper

In [ ]:
import pandas as pd, json
a = pd.read_csv('out/summary_2003_2008.csv').set_index(['model','features'])['auc'].round(3)
paper_a = {('logit','ratios14'):0.687, ('logit','ratios14_tax'):0.690, ('rusboost','raw26_notax'):0.714,
           ('rusboost','raw28'):0.714, ('rusboost','raw28_tax'):0.714, ('xgb','raw28'):0.715, ('xgb','raw28_tax'):0.693}
print('Table 4: AUC 2003-2008 (yours vs paper)')
for k, v in paper_a.items():
    print(f'  {k[0]:9} {k[1]:13} yours {a.get(k, float("nan")):.3f}   paper {v:.3f}')
b = pd.read_csv('extension_2009_2022/out/ext_summary_main_2015_2020.csv').set_index(['model','features'])['auc'].round(3)
paper_b = {('rusboost','raw27'):0.670, ('rusboost','raw27_tax'):0.677, ('rusboost','raw27_tax_full'):0.652,
           ('xgb','raw27'):0.638, ('xgb','raw27_tax'):0.614, ('xgb','raw27_tax_full'):0.622}
print('Table 6: AUC 2015-2020 (yours vs paper)')
for k, v in paper_b.items():
    print(f'  {k[0]:9} {k[1]:15} yours {b.get(k, float("nan")):.3f}   paper {v:.3f}')
s = json.load(open('out/shap_stability.json'))
print(f"SHAP stability (Spearman mean): yours {s['spearman_mean']:.2f}   paper 0.92")
print(f"Tax share of SHAP: yours {s['tax_share_of_total_shap']:.1%}   paper 15.5%")
sp = pd.read_csv('out/sparse_by_year.csv').groupby('k').auc.mean().round(3)
print(f"Top-15 sparse model AUC: yours {sp.get(15):.3f}   paper 0.718")